# 🚀 FASE 1 - Instalación de Docker en Google Colab

In [ ]:

!apt-get update
!apt install docker.io -y
!service docker start
!docker --version


# 🚀 FASE 2 - Inicializar Proyecto RASA

In [ ]:

!mkdir -p /content/rasa_project
%cd /content/rasa_project
!docker pull rasa/rasa:3.6.10
!docker run -v $(pwd):/app rasa/rasa:3.6.10 init --no-prompt


# 🚀 FASE 3 - Modificar el proyecto para agregar intents

In [ ]:

nlu_content = """
version: "3.1"
nlu:
- intent: saludo
  examples: |
    - hola
    - buenos días
    - qué tal

- intent: horario
  examples: |
    - ¿cuál es el horario del curso?
    - dime los horarios de clase

- intent: ubicacion
  examples: |
    - ¿dónde es el aula?
    - dime el aula del curso
"""

with open('/content/rasa_project/data/nlu.yml', 'w') as f:
    f.write(nlu_content)

domain_content = """
version: "3.1"
intents:
  - saludo
  - horario
  - ubicacion

responses:
  utter_saludo:
  - text: "¡Hola! ¿En qué puedo ayudarte con el curso de IA?"
  
  utter_horario:
  - text: "El horario del curso es lunes y miércoles de 17:00 a 19:00."

  utter_ubicacion:
  - text: "El aula es la 3.1 del edificio principal."

actions:
  - action_fallback
"""

with open('/content/rasa_project/domain.yml', 'w') as f:
    f.write(domain_content)

config_content = """
version: "3.1"
policies:
  - name: RulePolicy
    core_fallback_threshold: 0.4
    core_fallback_action_name: "action_fallback"
"""

with open('/content/rasa_project/config.yml', 'w') as f:
    f.write(config_content)


# 🚀 FASE 4 - Añadir acción personalizada LLM

In [ ]:

actions_content = """
from rasa_sdk import Action
from rasa_sdk.events import EventType
import openai

class ActionFallbackLLM(Action):
    def name(self) -> str:
        return "action_fallback"

    def run(self, dispatcher, tracker, domain) -> list[EventType]:
        user_message = tracker.latest_message.get('text')

        openai.api_key = "TU_API_KEY_AQUI"

        response = openai.Completion.create(
            engine="text-davinci-003",
            prompt=f"Responde como un asistente de un curso de IA. Pregunta: {user_message}",
            max_tokens=100
        )
        answer = response.choices[0].text.strip()

        dispatcher.utter_message(text=answer)
        return []
"""

with open('/content/rasa_project/actions/actions.py', 'w') as f:
    f.write(actions_content)

endpoints_content = """
action_endpoint:
  url: "http://localhost:5055/webhook"
"""

with open('/content/rasa_project/endpoints.yml', 'w') as f:
    f.write(endpoints_content)

rules_content = """
version: "3.1"
rules:
- rule: fallback to LLM
  steps:
  - intent: nlu_fallback
  - action: action_fallback
"""

with open('/content/rasa_project/data/rules.yml', 'w') as f:
    f.write(rules_content)


# 🚀 FASE 5 - Entrenar el modelo

In [ ]:

!docker run -v $(pwd):/app rasa/rasa:3.6.10 train


# 🚀 FASE 6 - Lanzar servidor de actions y RASA

In [ ]:

!docker run -v $(pwd):/app -p 5055:5055 rasa/rasa-sdk:3.6.10 &
!docker run -v $(pwd):/app -p 5005:5005 rasa/rasa:3.6.10 run --enable-api --cors "*" &


# 🚀 FASE 7 - Exponer puerto a internet con ngrok

In [ ]:

!pip install pyngrok
from pyngrok import ngrok
public_url = ngrok.connect(5005)
print("Tu agente RASA está accesible en la siguiente URL pública:")
print(public_url)
